In [1]:
%pip install pandas xlrd openpyxl sqlalchemy pymysql

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
import logging
import warnings

warnings.filterwarnings("ignore")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

print("Libraries imported successfully")

Libraries imported successfully


In [3]:
RAW_FILE = Path(r"D:\Bank_loan_project\data\default of credit card clients.xls")

OUTPUT_DIR = Path(r"D:\Bank_loan_project\notebook_processed_data")
OUTPUT_DIR.mkdir(exist_ok=True)

print("Input file :", RAW_FILE)
print("Output dir :", OUTPUT_DIR)

Input file : D:\Bank_loan_project\data\default of credit card clients.xls
Output dir : D:\Bank_loan_project\notebook_processed_data


In [4]:
excel_file = pd.ExcelFile(RAW_FILE, engine="xlrd")

print("Available sheets:")
print(excel_file.sheet_names)

Available sheets:
['Data']


In [5]:
df_raw = pd.read_excel(
    RAW_FILE,
    sheet_name="Data",
    header=1,
    engine="xlrd"
)

print("Dataset loaded successfully")
print("Shape:", df_raw.shape)

df_raw.head()

Dataset loaded successfully
Shape: (30000, 25)


,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
0,1,20000,2,2,1,24,2,2,-1,-1,...,0,0,0,0,689,0,0,0,0,1
1,2,120000,2,2,2,26,-1,2,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,3,90000,2,2,2,34,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,4,50000,2,2,1,37,0,0,0,0,...,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,5,50000,1,2,1,57,-1,0,-1,0,...,20940,19146,19131,2000,36681,10000,9000,689,679,0


In [6]:
print("Rows   :", df_raw.shape[0])
print("Columns:", df_raw.shape[1])

print("\nColumn names:")
print(df_raw.columns.tolist())

print("\nData types:")
print(df_raw.dtypes)

Rows   : 30000
Columns: 25

Column names:
['ID', 'LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE', 'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6', 'default payment next month']

Data types:
ID                            int64
LIMIT_BAL                     int64
SEX                           int64
EDUCATION                     int64
MARRIAGE                      int64
AGE                           int64
PAY_0                         int64
PAY_2                         int64
PAY_3                         int64
PAY_4                         int64
PAY_5                         int64
PAY_6                         int64
BILL_AMT1                     int64
BILL_AMT2                     int64
BILL_AMT3                     int64
BILL_AMT4                     int64
BILL_AMT5                     int64
BILL_AMT6                     int64
PAY_

In [7]:
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 25 columns):
 #   Column                      Non-Null Count  Dtype
---  ------                      --------------  -----
 0   ID                          30000 non-null  int64
 1   LIMIT_BAL                   30000 non-null  int64
 2   SEX                         30000 non-null  int64
 3   EDUCATION                   30000 non-null  int64
 4   MARRIAGE                    30000 non-null  int64
 5   AGE                         30000 non-null  int64
 6   PAY_0                       30000 non-null  int64
 7   PAY_2                       30000 non-null  int64
 8   PAY_3                       30000 non-null  int64
 9   PAY_4                       30000 non-null  int64
 10  PAY_5                       30000 non-null  int64
 11  PAY_6                       30000 non-null  int64
 12  BILL_AMT1                   30000 non-null  int64
 13  BILL_AMT2                   30000 non-null  int64
 14  BILL_AMT3        

In [8]:
df = df_raw.copy()

print("Working copy created")
print("Shape:", df.shape)

Working copy created
Shape: (30000, 25)


In [9]:
df.columns = (
    df.columns
      .astype(str)
      .str.strip()
      .str.lower()
      .str.replace(".", "_", regex=False)
      .str.replace(" ", "_", regex=False)
)

print(df.columns.tolist())

['id', 'limit_bal', 'sex', 'education', 'marriage', 'age', 'pay_0', 'pay_2', 'pay_3', 'pay_4', 'pay_5', 'pay_6', 'bill_amt1', 'bill_amt2', 'bill_amt3', 'bill_amt4', 'bill_amt5', 'bill_amt6', 'pay_amt1', 'pay_amt2', 'pay_amt3', 'pay_amt4', 'pay_amt5', 'pay_amt6', 'default_payment_next_month']


In [10]:
rename_map = {
    "limit_bal": "credit_limit",
    "sex": "gender",
    "pay_0": "repayment_status_sep",
    "pay_2": "repayment_status_aug",
    "pay_3": "repayment_status_jul",
    "pay_4": "repayment_status_jun",
    "pay_5": "repayment_status_may",
    "pay_6": "repayment_status_apr",
    "bill_amt1": "bill_amt_sep",
    "bill_amt2": "bill_amt_aug",
    "bill_amt3": "bill_amt_jul",
    "bill_amt4": "bill_amt_jun",
    "bill_amt5": "bill_amt_may",
    "bill_amt6": "bill_amt_apr",
    "pay_amt1": "payment_amt_sep",
    "pay_amt2": "payment_amt_aug",
    "pay_amt3": "payment_amt_jul",
    "pay_amt4": "payment_amt_jun",
    "pay_amt5": "payment_amt_may",
    "pay_amt6": "payment_amt_apr",
    "default_payment_next_month": "default_next_month"
}

df.rename(columns=rename_map, inplace=True)

df.head()

,id,credit_limit,gender,education,marriage,age,repayment_status_sep,repayment_status_aug,repayment_status_jul,repayment_status_jun,...,bill_amt_jun,bill_amt_may,bill_amt_apr,payment_amt_sep,payment_amt_aug,payment_amt_jul,payment_amt_jun,payment_amt_may,payment_amt_apr,default_next_month
0,1,20000,2,2,1,24,2,2,-1,-1,...,0,0,0,0,689,0,0,0,0,1
1,2,120000,2,2,2,26,-1,2,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,3,90000,2,2,2,34,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,4,50000,2,2,1,37,0,0,0,0,...,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,5,50000,1,2,1,57,-1,0,-1,0,...,20940,19146,19131,2000,36681,10000,9000,689,679,0


In [11]:
quality_report = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str).values,
    "missing_count": df.isnull().sum().values,
    "missing_percent": (df.isnull().mean() * 100).round(2).values,
    "unique_values": df.nunique().values
})

quality_report

,column,dtype,missing_count,missing_percent,unique_values
0,id,int64,0,0.0,30000
1,credit_limit,int64,0,0.0,81
2,gender,int64,0,0.0,2
3,education,int64,0,0.0,7
4,marriage,int64,0,0.0,4
5,age,int64,0,0.0,56
6,repayment_status_sep,int64,0,0.0,11
7,repayment_status_aug,int64,0,0.0,11
8,repayment_status_jul,int64,0,0.0,11
9,repayment_status_jun,int64,0,0.0,11


In [55]:
print("unique genders ", df['gender'].unique())
print("unique education ", df['education'].unique())
print("unique marriage ", df['marriage'].unique())
print("unique repayment vals ", df['repayment_status_aug'].unique())
	

unique genders  [2 1]
unique education  [2 1 3 5 4 6 0]
unique marriage  [1 2 3 0]
unique repayment vals  [ 2  0 -1 -2  3  5  7  4  1  6  8]


In [13]:
print("Duplicate rows:", df.duplicated().sum())

print(
    "Duplicate customer IDs:",
    df["id"].duplicated().sum()
)

Duplicate rows: 0
Duplicate customer IDs: 0


In [14]:
before = len(df)

df = df.drop_duplicates()
df = df.drop_duplicates(subset=["id"], keep="first")

after = len(df)

print("Rows before:", before)
print("Rows after :", after)
print("Removed    :", before - after)

Rows before: 30000
Rows after : 30000
Removed    : 0


In [15]:
missing = df.isnull().sum()

missing = missing[missing > 0].sort_values(ascending=False)

if len(missing) == 0:
    print("No missing values found")
else:
    print(missing)

No missing values found


In [16]:
numeric_columns = [
    "id",
    "credit_limit",
    "gender",
    "education",
    "marriage",
    "age",

    "repayment_status_sep",
    "repayment_status_aug",
    "repayment_status_jul",
    "repayment_status_jun",
    "repayment_status_may",
    "repayment_status_apr",

    "bill_amt_sep",
    "bill_amt_aug",
    "bill_amt_jul",
    "bill_amt_jun",
    "bill_amt_may",
    "bill_amt_apr",

    "payment_amt_sep",
    "payment_amt_aug",
    "payment_amt_jul",
    "payment_amt_jun",
    "payment_amt_may",
    "payment_amt_apr",

    "default_next_month"
]

for col in numeric_columns:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

print("Numeric conversion completed")

Numeric conversion completed


In [17]:
invalid_id = df[
    df["id"].isna() |
    (df["id"] <= 0)
]

print("Invalid customer IDs:", len(invalid_id))

Invalid customer IDs: 0


In [18]:
df = df[
    df["id"].notna() &
    (df["id"] > 0)
].copy()

df["id"] = df["id"].astype(int)

print("Remaining rows:", len(df))

Remaining rows: 30000


In [19]:
invalid_age = df[
    df["age"].isna() |
    (df["age"] < 18) |
    (df["age"] > 100)
]

print("Invalid age records:", len(invalid_age))

invalid_age[["id", "age"]].head()

Invalid age records: 0


,id,age


In [35]:
df["age_valid"] = df["age"].between(18, 100)

df["age_valid"].value_counts()

age_valid
True    30000
Name: count, dtype: int64

In [20]:
df["credit_limit_valid"] = (
    df["credit_limit"].notna() &
    (df["credit_limit"] > 0)
)

print(df["credit_limit_valid"].value_counts())

credit_limit_valid
True    30000
Name: count, dtype: int64


In [21]:
gender_map = {
    1: "Male",
    2: "Female"
}

education_map = {
    1: "Graduate School",
    2: "University",
    3: "High School",
    4: "Other",
    0: "Other",
    5: "Other",
    6: "Other"
}

marriage_map = {
    1: "Married",
    2: "Single",
    3: "Other",
    0: "Other"
}

df["gender_label"] = df["gender"].map(gender_map).fillna("Unknown")
df["education_label"] = df["education"].map(education_map).fillna("Unknown")
df["marriage_label"] = df["marriage"].map(marriage_map).fillna("Unknown")

df[
    [
        "gender", "gender_label",
        "education", "education_label",
        "marriage", "marriage_label"
    ]
].head()

,gender,gender_label,education,education_label,marriage,marriage_label
0,2,Female,2,University,1,Married
1,2,Female,2,University,2,Single
2,2,Female,2,University,2,Single
3,2,Female,2,University,1,Married
4,1,Male,2,University,1,Married


In [22]:
print(df["default_next_month"].value_counts(dropna=False))

default_next_month
0    23364
1     6636
Name: count, dtype: int64


In [23]:
df["default_valid"] = df["default_next_month"].isin([0, 1])

print(df["default_valid"].value_counts())

default_valid
True    30000
Name: count, dtype: int64


In [24]:
repayment_columns = [
    "repayment_status_sep",
    "repayment_status_aug",
    "repayment_status_jul",
    "repayment_status_jun",
    "repayment_status_may",
    "repayment_status_apr"
]

df[repayment_columns].describe()

,repayment_status_sep,repayment_status_aug,repayment_status_jul,repayment_status_jun,repayment_status_may,repayment_status_apr
count,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000
mean,-0.016700,-0.133767,-0.166200,-0.220667,-0.266200,-0.291100
std,1.123802,1.197186,1.196868,1.169139,1.133187,1.149988
min,-2.000000,-2.000000,-2.000000,-2.000000,-2.000000,-2.000000
25%,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000
50%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
max,8.000000,8.000000,8.000000,8.000000,8.000000,8.000000


In [25]:
bill_columns = [
    "bill_amt_sep",
    "bill_amt_aug",
    "bill_amt_jul",
    "bill_amt_jun",
    "bill_amt_may",
    "bill_amt_apr"
]

payment_columns = [
    "payment_amt_sep",
    "payment_amt_aug",
    "payment_amt_jul",
    "payment_amt_jun",
    "payment_amt_may",
    "payment_amt_apr"
]

df[bill_columns + payment_columns].describe().T

,count,mean,std,min,25%,50%,75%,max
bill_amt_sep,30000.0,51223.330900,73635.860576,-165580.0,3558.75,22381.5,67091.00,964511.0
bill_amt_aug,30000.0,49179.075167,71173.768783,-69777.0,2984.75,21200.0,64006.25,983931.0
bill_amt_jul,30000.0,47013.154800,69349.387427,-157264.0,2666.25,20088.5,60164.75,1664089.0
bill_amt_jun,30000.0,43262.948967,64332.856134,-170000.0,2326.75,19052.0,54506.00,891586.0
bill_amt_may,30000.0,40311.400967,60797.155770,-81334.0,1763.00,18104.5,50190.50,927171.0
bill_amt_apr,30000.0,38871.760400,59554.107537,-339603.0,1256.00,17071.0,49198.25,961664.0
payment_amt_sep,30000.0,5663.580500,16563.280354,0.0,1000.00,2100.0,5006.00,873552.0
payment_amt_aug,30000.0,5921.163500,23040.870402,0.0,833.00,2009.0,5000.00,1684259.0
payment_amt_jul,30000.0,5225.681500,17606.961470,0.0,390.00,1800.0,4505.00,896040.0
payment_amt_jun,30000.0,4826.076867,15666.159744,0.0,296.00,1500.0,4013.25,621000.0


In [26]:
#Validate payment amounts-Previous payment amounts should not be negative.
for col in payment_columns:
    df[col + "_valid"] = df[col].fillna(-1) >= 0

payment_valid_columns = [
    col + "_valid" for col in payment_columns
]

df["payments_valid"] = df[payment_valid_columns].all(axis=1)

print(df["payments_valid"].value_counts())

payments_valid
True    30000
Name: count, dtype: int64


In [27]:
df["age_group"] = pd.cut(
    df["age"],
    bins=[17, 25, 35, 45, 55, 65, 100],
    labels=[
        "18-25",
        "26-35",
        "36-45",
        "46-55",
        "56-65",
        "65+"
    ]
)

df["age_group"].value_counts().sort_index()

age_group
18-25     3871
26-35    12938
36-45     8522
46-55     3825
56-65      758
65+         86
Name: count, dtype: int64

In [28]:
df["total_bill_amount"] = df[bill_columns].sum(axis=1)

df[
    ["id", "total_bill_amount"]
].head()

,id,total_bill_amount
0,1,7704
1,2,17077
2,3,101653
3,4,231334
4,5,109339


In [29]:
df["total_payment_amount"] = df[payment_columns].sum(axis=1)

df[
    ["id", "total_payment_amount"]
].head()

,id,total_payment_amount
0,1,689
1,2,5000
2,3,11018
3,4,8388
4,5,59049


In [30]:
df["avg_bill_amount"] = df[bill_columns].mean(axis=1)

df["avg_payment_amount"] = df[payment_columns].mean(axis=1)

df[
    [
        "id",
        "avg_bill_amount",
        "avg_payment_amount"
    ]
].head()

,id,avg_bill_amount,avg_payment_amount
0,1,1284.000000,114.833333
1,2,2846.166667,833.333333
2,3,16942.166667,1836.333333
3,4,38555.666667,1398.000000
4,5,18223.166667,9841.500000


In [31]:
df["latest_credit_utilization"] = np.where(
    df["credit_limit"] > 0,
    df["bill_amt_sep"] / df["credit_limit"],
    np.nan
)

df[
    [
        "id",
        "credit_limit",
        "bill_amt_sep",
        "latest_credit_utilization"
    ]
].head()

,id,credit_limit,bill_amt_sep,latest_credit_utilization
0,1,20000,3913,0.195650
1,2,120000,2682,0.022350
2,3,90000,29239,0.324878
3,4,50000,46990,0.939800
4,5,50000,8617,0.172340


In [32]:
df["months_with_delay"] = (
    df[repayment_columns] > 0
).sum(axis=1)

df["max_repayment_delay"] = df[repayment_columns].max(axis=1)

df[
    [
        "id",
        "months_with_delay",
        "max_repayment_delay"
    ]
].head()

,id,months_with_delay,max_repayment_delay
0,1,2,2
1,2,2,2
2,3,0,0
3,4,0,0
4,5,0,0


In [33]:
df["payment_to_bill_ratio"] = np.where(
    df["total_bill_amount"] > 0,
    df["total_payment_amount"] / df["total_bill_amount"],
    np.nan
)

df[
    [
        "id",
        "total_bill_amount",
        "total_payment_amount",
        "payment_to_bill_ratio"
    ]
].head()

,id,total_bill_amount,total_payment_amount,payment_to_bill_ratio
0,1,7704,689,0.089434
1,2,17077,5000,0.292791
2,3,101653,11018,0.108388
3,4,231334,8388,0.036259
4,5,109339,59049,0.540054


In [36]:
df["data_quality_valid"] = (
    df["age_valid"] &
    df["credit_limit_valid"] &
    df["default_valid"] &
    df["payments_valid"]
)

df["data_quality_valid"].value_counts()


data_quality_valid
True    30000
Name: count, dtype: int64

In [37]:
rejected_df = df[
    ~df["data_quality_valid"]
].copy()

curated_df = df[
    df["data_quality_valid"]
].copy()

print("Total records   :", len(df))
print("Valid records   :", len(curated_df))
print("Rejected records:", len(rejected_df))

Total records   : 30000
Valid records   : 30000
Rejected records: 0


In [38]:
def get_rejection_reason(row):
    reasons = []

    if not row["age_valid"]:
        reasons.append("INVALID_AGE")

    if not row["credit_limit_valid"]:
        reasons.append("INVALID_CREDIT_LIMIT")

    if not row["default_valid"]:
        reasons.append("INVALID_DEFAULT_VALUE")

    if not row["payments_valid"]:
        reasons.append("INVALID_PAYMENT_AMOUNT")

    return ",".join(reasons)


rejected_df["rejection_reason"] = rejected_df.apply(
    get_rejection_reason,
    axis=1
)

rejected_df[
    ["id", "rejection_reason"]
].head()

,id,rejection_reason


In [39]:
etl_summary = {
    "raw_records": len(df_raw),
    "records_after_duplicate_removal": len(df),
    "valid_records": len(curated_df),
    "rejected_records": len(rejected_df),
    "duplicate_rows_in_source": int(df_raw.duplicated().sum()),
    "missing_values": int(df_raw.isnull().sum().sum())
}

for key, value in etl_summary.items():
    print(f"{key}: {value}")

raw_records: 30000
records_after_duplicate_removal: 30000
valid_records: 30000
rejected_records: 0
duplicate_rows_in_source: 0
missing_values: 0


In [40]:
customers_df = curated_df[
    [
        "id",
        "gender",
        "gender_label",
        "education",
        "education_label",
        "marriage",
        "marriage_label",
        "age",
        "age_group"
    ]
].copy()

customers_df.rename(
    columns={"id": "customer_id"},
    inplace=True
)

customers_df.head()

,customer_id,gender,gender_label,education,education_label,marriage,marriage_label,age,age_group
0,1,2,Female,2,University,1,Married,24,18-25
1,2,2,Female,2,University,2,Single,26,26-35
2,3,2,Female,2,University,2,Single,34,26-35
3,4,2,Female,2,University,1,Married,37,36-45
4,5,1,Male,2,University,1,Married,57,56-65


In [41]:
credit_profiles_df = curated_df[
    [
        "id",
        "credit_limit",
        "repayment_status_sep",
        "repayment_status_aug",
        "repayment_status_jul",
        "repayment_status_jun",
        "repayment_status_may",
        "repayment_status_apr",
        "months_with_delay",
        "max_repayment_delay",
        "total_bill_amount",
        "total_payment_amount",
        "avg_bill_amount",
        "avg_payment_amount",
        "latest_credit_utilization",
        "payment_to_bill_ratio",
        "default_next_month"
    ]
].copy()

credit_profiles_df.rename(
    columns={"id": "customer_id"},
    inplace=True
)

credit_profiles_df.head()

,customer_id,credit_limit,repayment_status_sep,repayment_status_aug,repayment_status_jul,repayment_status_jun,repayment_status_may,repayment_status_apr,months_with_delay,max_repayment_delay,total_bill_amount,total_payment_amount,avg_bill_amount,avg_payment_amount,latest_credit_utilization,payment_to_bill_ratio,default_next_month
0,1,20000,2,2,-1,-1,-2,-2,2,2,7704,689,1284.000000,114.833333,0.195650,0.089434,1
1,2,120000,-1,2,0,0,0,2,2,2,17077,5000,2846.166667,833.333333,0.022350,0.292791,1
2,3,90000,0,0,0,0,0,0,0,0,101653,11018,16942.166667,1836.333333,0.324878,0.108388,0
3,4,50000,0,0,0,0,0,0,0,0,231334,8388,38555.666667,1398.000000,0.939800,0.036259,0
4,5,50000,-1,0,-1,0,0,0,0,0,109339,59049,18223.166667,9841.500000,0.172340,0.540054,0


In [42]:
bill_months = {
    "bill_amt_apr": "APR",
    "bill_amt_may": "MAY",
    "bill_amt_jun": "JUN",
    "bill_amt_jul": "JUL",
    "bill_amt_aug": "AUG",
    "bill_amt_sep": "SEP"
}

bill_frames = []

for column, month in bill_months.items():

    temp = curated_df[
        ["id", column]
    ].copy()

    temp.columns = [
        "customer_id",
        "bill_amount"
    ]

    temp["month"] = month

    bill_frames.append(temp)


bills_df = pd.concat(
    bill_frames,
    ignore_index=True
)

bills_df.head(10)

,customer_id,bill_amount,month
0,1,0,APR
1,2,3261,APR
2,3,15549,APR
3,4,29547,APR
4,5,19131,APR
5,6,20024,APR
6,7,473944,APR
7,8,567,APR
8,9,3719,APR
9,10,13912,APR


In [43]:
payment_months = {
    "payment_amt_apr": "APR",
    "payment_amt_may": "MAY",
    "payment_amt_jun": "JUN",
    "payment_amt_jul": "JUL",
    "payment_amt_aug": "AUG",
    "payment_amt_sep": "SEP"
}

payment_frames = []

for column, month in payment_months.items():

    temp = curated_df[
        ["id", column]
    ].copy()

    temp.columns = [
        "customer_id",
        "payment_amount"
    ]

    temp["month"] = month

    payment_frames.append(temp)


payments_df = pd.concat(
    payment_frames,
    ignore_index=True
)

payments_df.head(10)

,customer_id,payment_amount,month
0,1,0,APR
1,2,2000,APR
2,3,5000,APR
3,4,1000,APR
4,5,679,APR
5,6,800,APR
6,7,13770,APR
7,8,1542,APR
8,9,1000,APR
9,10,0,APR


In [44]:
print("Customers       :", customers_df.shape)
print("Credit Profiles :", credit_profiles_df.shape)
print("Bills           :", bills_df.shape)
print("Payments        :", payments_df.shape)
print("Rejected        :", rejected_df.shape)

Customers       : (30000, 9)
Credit Profiles : (30000, 17)
Bills           : (180000, 3)
Payments        : (180000, 3)
Rejected        : (0, 49)


In [45]:
assert len(bills_df) == len(customers_df) * 6
assert len(payments_df) == len(customers_df) * 6

print("Relational ETL validation passed")

Relational ETL validation passed


In [46]:
customers_df.to_csv(
    OUTPUT_DIR / "customers.csv",
    index=False
)

credit_profiles_df.to_csv(
    OUTPUT_DIR / "credit_profiles.csv",
    index=False
)

bills_df.to_csv(
    OUTPUT_DIR / "bills.csv",
    index=False
)

payments_df.to_csv(
    OUTPUT_DIR / "payments.csv",
    index=False
)

rejected_df.to_csv(
    OUTPUT_DIR / "rejected_records.csv",
    index=False
)

print("Processed files saved successfully")

Processed files saved successfully
